# Régression linéaire multiple — Cas confirmés de Mpox (RDC)

Notebook d'analyse reproductible. Il s'appuie sur les modules du dossier `src/`.
Pour régénérer l'ensemble des résultats et figures, on peut aussi lancer `python main.py`.

In [1]:
# Permet d'importer le package src/ que le notebook soit lancé
# depuis la racine du projet ou depuis le dossier notebooks/.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Racine du projet :", ROOT)

Racine du projet : /home/claude/mpox_multiple_regression


In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
np.random.seed(42)

from src import data_preprocessing as prep
from src import exploratory_analysis as eda
from src import regression_model as reg
from src import diagnostics as diag
from src import model_evaluation as evaluate
from src.regression_model import _clean_names
import statsmodels.api as sm

## 1. Prétraitement et contrôle de la fuite de données

On vérifie l'identité `Taux_Positivite_pct = Cas_Confirmes / Tests_Realises × 100`,
qui impose d'exclure `Taux_Positivite_pct` des prédicteurs.

In [3]:
df = prep.preprocess(save=True)
print("Dimensions :", df.shape)
print("Doublons :", prep.check_duplicates(df))
print("Écart max identité de fuite :", prep.verify_leakage_identity(df))
df.head()

Dimensions : (3000, 17)
Doublons : {'lignes_dupliquees': 0, 'id_dupliques': 0}
Écart max identité de fuite : 0.0


,ID,Semaine,Province,Pluviometrie_mm,Temperature_C,NDVI,Humidite_pct,Densite_Population,Couverture_Vaccinale_pct,Tests_Realises,Distance_Centre_Sante_km,Reservoirs_Animaux,Mobilite_Humaine,Cas_Confirmes,Saison,Population_Risque,Taux_Positivite_pct
0,OBS_00001,1,Mongala,206,24.4,0.480,64,39,31.8,150,22,1,4,29,Pluie,3478,19.33
1,OBS_00002,2,Kinshasa,56,23.7,0.474,74,107,24.9,300,5,1,6,42,Seche,3713,14.00
2,OBS_00003,3,Kinshasa,59,25.2,0.333,62,140,53.9,310,16,0,8,20,Seche,4833,6.45
3,OBS_00004,4,Mai-Ndombe,396,24.1,0.354,68,97,25.9,210,4,0,10,82,Pluie,3777,39.05
4,OBS_00005,5,Kasaï,127,19.8,0.303,61,86,30.3,260,24,1,6,24,Pluie,9396,9.23


## 2. Analyse exploratoire

In [4]:
eda.descriptive_statistics(df, save=False).head(10)

,count,mean,std,min,25%,50%,75%,max,mediane,variance,cv
Pluviometrie_mm,3000.0,157.780,101.959,2.000,77.000,136.000,221.000,400.000,136.000,10395.592,0.646
Temperature_C,3000.0,24.980,3.833,18.000,22.300,24.900,27.600,34.000,24.900,14.688,0.153
NDVI,3000.0,0.451,0.158,0.103,0.331,0.452,0.576,0.795,0.452,0.025,0.350
Humidite_pct,3000.0,71.572,11.625,40.000,64.000,71.500,80.000,95.000,71.500,135.147,0.162
Densite_Population,3000.0,119.100,95.218,5.000,48.000,94.000,163.000,500.000,94.000,9066.555,0.799
Couverture_Vaccinale_pct,3000.0,31.844,14.080,5.600,20.800,30.400,41.525,77.800,30.400,198.233,0.442
Tests_Realises,3000.0,286.143,88.256,90.000,220.000,280.000,340.000,800.000,280.000,7789.156,0.308
Distance_Centre_Sante_km,3000.0,15.558,15.139,1.000,5.000,11.000,22.000,142.000,11.000,229.201,0.973
Mobilite_Humaine,3000.0,5.018,1.954,1.000,4.000,5.000,6.000,10.000,5.000,3.819,0.389
Population_Risque,3000.0,5471.298,2636.188,1001.000,3127.000,5514.500,7819.250,9997.000,5514.500,6949487.852,0.482


In [5]:
print("Corrélations avec la cible :")
eda.target_correlations(df)

Corrélations avec la cible :


Pluviometrie_mm             0.625
Tests_Realises              0.462
Densite_Population          0.182
Temperature_C               0.131
NDVI                        0.092
Humidite_pct                0.043
Mobilite_Humaine            0.025
Population_Risque           0.023
Reservoirs_Animaux         -0.040
Distance_Centre_Sante_km   -0.154
Couverture_Vaccinale_pct   -0.350
Name: Cas_Confirmes, dtype: float64

## 3. Séparation et modèle OLS complet

Séparation aléatoire 80/20 justifiée : données transversales (`Semaine` = index).

In [6]:
data = reg.build_design_matrix(df)
X_train, X_test, y_train, y_test = reg.split_data(data, test_size=0.2)
ols = reg.fit_ols(X_train, y_train)
print(f"R2={ols.rsquared:.3f}  R2 ajuste={ols.rsquared_adj:.3f}  AIC={ols.aic:.0f}  BIC={ols.bic:.0f}")

R2=0.824  R2 ajuste=0.821  AIC=23064  BIC=23284


## 4. Modèle final (sélection backward AIC)

In [7]:
retained, final = reg.backward_selection_aic(X_train, y_train)
print(f"{len(retained)} variables retenues")
print(f"R2={final.rsquared:.3f}  R2 ajuste={final.rsquared_adj:.3f}  AIC={final.aic:.0f}  BIC={final.bic:.0f}")
final.summary()

14 variables retenues
R2=0.823  R2 ajuste=0.822  AIC=23029  BIC=23116


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          Cas_Confirmes   R-squared:                       0.823
Model:                            OLS   Adj. R-squared:                  0.822
Method:                 Least Squares   F-statistic:                     791.9
Date:                Mon, 20 Jul 2026   Prob (F-statistic):               0.00
Time:                        22:50:13   Log-Likelihood:                -11500.
No. Observations:                2400   AIC:                         2.303e+04
Df Residuals:                    2385   BIC:                         2.312e+04
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                     -164.7826      6.556    -25.133      0.000    -177.639    -151.926
Pluviometrie_mm              0.4508      0.008     53.671      0.000       0.434       0.467
Temperature_C                2.6179      0.157     16.656      0.000       2.310       2.926
NDVI                        60.1010      3.797     15.830      0.000      52.656      67.546
Humidite_pct                 0.3771      0.051      7.332      0.000       0.276       0.478
Densite_Population           0.1425      0.006     22.923      0.000       0.130       0.155
Couverture_Vaccinale_pct    -1.7230      0.042    -40.573      0.000      -1.806      -1.640
Tests_Realises               0.3607      0.007     52.621      0.000       0.347       0.374
Distance_Centre_Sante_km    -0.8175      0.040    -20.582      0.000      -0.895      -0.740
Reservoirs_Animaux          -2.9066      1.222     -2.378      0.017      -5.303      -0.510
Province_Haut_Uele          -5.3603      2.984     -1.796      0.073     -11.212       0.492
Province_Kasaï_Central      -4.9903      3.210     -1.554      0.120     -11.286       1.305
Province_Lomami              4.4783      3.001      1.492      0.136      -1.406      10.363
Province_Sankuru            -7.2854      3.062     -2.379      0.017     -13.290      -1.281
Saison_Seche                 7.1992      1.790      4.023      0.000       3.690      10.709
==============================================================================
Omnibus:                      473.923   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1304.171
Skew:                           1.037   Prob(JB):                    6.35e-284
Kurtosis:                       5.957   Cond. No.                     4.12e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.12e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## 5. Diagnostics des hypothèses

In [8]:
X_final = _clean_names(X_train)[retained].astype(float)
Xc = sm.add_constant(X_final)
print("Normalité       :", diag.normality_tests(final))
print("Homoscédasticité:", diag.homoscedasticity_test(final, Xc))
print("Indépendance    :", diag.independence_test(final))
print("RESET linéarité :", diag.linearity_reset(final))
print("Influence       :", diag.influence_summary(final, len(X_train)))

Normalité       : {'shapiro_stat': np.float64(0.9513), 'shapiro_p': np.float64(0.0), 'jarque_bera_stat': np.float64(1304.1712), 'jarque_bera_p': np.float64(0.0), 'skewness': np.float64(1.0369), 'kurtosis': np.float64(5.9565)}
Homoscédasticité: {'breusch_pagan_LM': np.float64(300.2711), 'breusch_pagan_p': np.float64(0.0), 'f_stat': np.float64(24.3619), 'f_p': np.float64(0.0)}
Indépendance    : {'durbin_watson': np.float64(2.0175)}
RESET linéarité : {'reset_F': 2206.5654, 'reset_p': 0.0}


Influence       : {'seuil_cook': 0.001667, 'n_obs_influentes': 135, 'cook_max': 0.0283}


In [9]:
from src import NUMERIC_FEATURES
diag.vif_table(df[NUMERIC_FEATURES], save=False)

,variable,VIF,tolerance
2,NDVI,1.006,0.9944
5,Couverture_Vaccinale_pct,1.005,0.9953
4,Densite_Population,1.005,0.9953
6,Tests_Realises,1.005,0.9949
0,Pluviometrie_mm,1.004,0.9960
7,Distance_Centre_Sante_km,1.004,0.9956
1,Temperature_C,1.003,0.9969
3,Humidite_pct,1.003,0.9975
9,Population_Risque,1.003,0.9969
8,Mobilite_Humaine,1.002,0.9980


### Erreurs standards robustes (HC3)

L'hétéroscédasticité détectée est traitée par des erreurs robustes ; les
conclusions restent inchangées.

In [10]:
robust = final.get_robustcov_results(cov_type="HC3")
pd.DataFrame({"coef": final.params, "p_classique": final.pvalues,
              "p_robuste_HC3": robust.pvalues}).round(4)

,coef,p_classique,p_robuste_HC3
const,-164.7826,0.0000,0.0000
Pluviometrie_mm,0.4508,0.0000,0.0000
Temperature_C,2.6179,0.0000,0.0000
NDVI,60.1010,0.0000,0.0000
Humidite_pct,0.3771,0.0000,0.0000
Densite_Population,0.1425,0.0000,0.0000
Couverture_Vaccinale_pct,-1.7230,0.0000,0.0000
Tests_Realises,0.3607,0.0000,0.0000
Distance_Centre_Sante_km,-0.8175,0.0000,0.0000
Reservoirs_Animaux,-2.9066,0.0175,0.0170


## 6. Évaluation prédictive et validation croisée

In [11]:
Xtr = _clean_names(X_train)[retained]; Xte = _clean_names(X_test)[retained]
metrics = evaluate.evaluate_ols(final, Xtr, y_train, Xte, y_test, save=False)
print(metrics.to_string(index=False))
print("\nValidation croisée :", evaluate.cross_validation(
    _clean_names(data.drop(columns=['Cas_Confirmes'])), data['Cas_Confirmes'], cv=10))

          jeu     MSE   RMSE    MAE     R2
apprentissage 849.994 29.155 21.674 0.8230
         test 933.184 30.548 22.313 0.8258

Validation croisée : {'cv_folds': 10, 'r2_moyen': 0.8197, 'r2_ecart_type': 0.0131, 'r2_par_pli': [0.8408, 0.8269, 0.8133, 0.8152, 0.8004, 0.81, 0.8133, 0.8375, 0.8327, 0.8068]}


## 7. Modèles complémentaires (comptage et régularisation)

In [12]:
counts = reg.fit_count_models(df)
print(f"Poisson AIC = {counts['poisson'].aic:.0f}  |  Binomial négatif AIC = {counts['negbin'].aic:.0f}")
reg.fit_regularized(X_train, y_train, X_test, y_test)

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Poisson AIC = 34686  |  Binomial négatif AIC = 30379


,modele,R2_test
0,Ridge,0.8248
1,LASSO,0.8251
2,ElasticNet,0.8242


## Conclusion

Le modèle final (14 variables) explique ~82 % de la variance des cas confirmés,
avec une bonne généralisation. Voir `reports/report.md` pour l'interprétation
complète et `figures/` pour les graphiques.